<a href="https://colab.research.google.com/github/Ali-00/AWS-GLUE-ETL/blob/main/Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [2]:
!pip install xgboost
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.2 MB/s eta 0:00:00


In [3]:
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False

try:
    from catboost import CatBoostClassifier
    CAT_AVAILABLE = True
except:
    CAT_AVAILABLE = False

In [5]:
DATA_PATH = "/content/drive/MyDrive/Assignment_Data.csv"

df = pd.read_csv(DATA_PATH, sep=";")

print("\n" + "="*80)
print("DATA SHAPE:", df.shape)


DATA SHAPE: (2572, 55)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2572 entries, 0 to 2571
Data columns (total 55 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   country_id                2572 non-null   int64  
 1   application_id            2572 non-null   int64  
 2   product_id                2572 non-null   int64  
 3   Variable_1                2572 non-null   int64  
 4   Variable_2                2572 non-null   int64  
 5   Variable_3                2572 non-null   int64  
 6   Variable_4                2572 non-null   int64  
 7   Variable_5                2572 non-null   object 
 8   Variable_6                2537 non-null   float64
 9   Variable_7                2571 non-null   float64
 10  due_date                  2572 non-null   object 
 11  first_status_day_date     2572 non-null   object 
 12  first_status_time_of_day  2572 non-null   object 
 13  paid_date                 2572 non-null   object 
 14  Variable

In [11]:
missing_df = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': (df.isnull().mean() * 100).round(2)
})

missing_df = missing_df.sort_values(
    by='missing_percent',
    ascending=False
)

missing_df

,missing_count,missing_percent
Variable_42,2463,95.76
Variable_43,2424,94.25
Variable_44,2371,92.19
Variable_21,1990,77.37
Variable_38,1990,77.37
Variable_22,1757,68.31
Variable_39,1757,68.31
Variable_20,1100,42.77
Variable_37,1100,42.77
Target,515,20.02


In [16]:
df.groupby('Target')['Variable_44'].apply(
    lambda x: x.isna().mean()
)

,Variable_44
Target,
0.0,0.898894
1.0,0.924860


In [14]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
df.head(2)

,country_id,application_id,product_id,Variable_1,Variable_2,Variable_3,Variable_4,Variable_5,Variable_6,Variable_7,due_date,first_status_day_date,first_status_time_of_day,paid_date,Variable_8,customer_id,arrived_date,Variable_9,Variable_10,Variable_11,Variable_12,Variable_13,Variable_14,Variable_15,Variable_16,Variable_17,Variable_18,Variable_19,Variable_20,Variable_21,Variable_22,Variable_23,Variable_24,Variable_25,Variable_26,Variable_27,Variable_28,Variable_29,Variable_30,Variable_31,Variable_32,Variable_33,Variable_34,Variable_35,Variable_36,Variable_37,Variable_38,Variable_39,Variable_40,Variable_41,Variable_42,Variable_43,Variable_44,Variable_45,Target
0,21,24176,21210001,35,1,1,1,N,72.0,27.0,07/07/2015,01/06/2015,00:43:22,02/06/2015,100,2970192,01/06/2015 00:43,190,100,100,15.0,M,RATINGSTUFE M,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,14/06/2013,F,1.0
1,21,24185,21210001,30,1,1,1,N,25.0,26.0,02/07/2015,01/06/2015,07:56:36,02/06/2015,199,2970195,01/06/2015 07:56,199,199,199,15.0,M,RATINGSTUFE M,4.0,1.0,0.0,0.0,0.0,25000.0,NaN,100.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40147.0,NaN,100.0,0.0,0.0,NaN,NaN,NaN,M,1.0


In [10]:
df.describe()

,country_id,application_id,product_id,Variable_1,Variable_2,Variable_3,Variable_4,Variable_6,Variable_7,Variable_8,customer_id,Variable_9,Variable_10,Variable_11,Variable_12,Variable_15,Variable_16,Variable_17,Variable_18,Variable_19,Variable_20,Variable_21,Variable_22,Variable_23,Variable_24,Variable_25,Variable_26,Variable_27,Variable_28,Variable_29,Variable_30,Variable_31,Variable_32,Variable_33,Variable_34,Variable_35,Variable_36,Variable_37,Variable_38,Variable_39,Variable_40,Variable_41,Target
count,2572.0,2572.000000,2572.0,2572.000000,2572.000000,2572.0,2572.000000,2537.000000,2571.000000,2572.000000,2.572000e+03,2572.000000,2572.000000,2572.000000,2572.000000,2388.000000,2388.000000,2388.000000,2388.000000,2388.000000,1472.000000,582.000000,815.000000,2388.000000,2388.000000,2388.000000,2388.000000,2388.000000,2388.0,2388.000000,2388.0,2388.0,2388.0,2388.0,2388.0,2388.000000,2388.000000,1472.000000,582.000000,815.000000,2388.000000,2388.000000,2057.000000
mean,21.0,63414.942457,21210001.0,33.641913,1.301322,1.0,1.000778,49.090658,32.246597,193.284992,3.965643e+06,230.280327,200.349922,200.349922,38.392885,2.152848,1.164154,0.328727,0.575377,0.429229,14834.296875,4333.276632,2296.866258,0.143635,0.037688,0.021357,0.340034,1.840452,0.0,0.064489,0.0,0.0,0.0,0.0,0.0,0.087940,0.025544,32898.739810,5420.609966,3471.501840,0.184255,0.004188,0.692270
std,0.0,22113.941599,0.0,9.089623,0.714760,0.0,0.027880,26.176144,11.531376,110.996680,6.733712e+05,114.915400,78.806132,78.806132,27.932548,2.679439,2.922519,0.678520,0.881650,0.939083,13868.911178,6116.112136,6653.736428,0.556687,0.207331,0.155759,0.687397,3.326539,0.0,0.381862,0.0,0.0,0.0,0.0,0.0,0.416226,0.157805,41066.489101,8791.836300,7977.484235,0.446054,0.064590,0.461666
min,21.0,24176.000000,21210001.0,7.000000,1.000000,1.0,1.000000,1.000000,18.000000,0.000000,2.345648e+06,50.000000,50.000000,50.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,300.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,100.000000,300.000000,100.000000,0.000000,0.000000,0.000000
25%,21.0,44761.500000,21210001.0,30.000000,1.000000,1.0,1.000000,27.000000,24.000000,100.000000,3.342499e+06,190.000000,190.000000,190.000000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4000.000000,1201.250000,200.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,5000.000000,1300.000000,300.000000,0.000000,0.000000,0.000000
50%,21.0,66149.000000,21210001.0,30.000000,1.000000,1.0,1.000000,49.000000,29.000000,199.000000,4.088723e+06,199.000000,199.000000,199.000000,45.000000,1.000000,0.000000,0.000000,0.000000,0.000000,10909.500000,3000.000000,450.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,17321.000000,3000.000000,639.000000,0.000000,0.000000,1.000000
75%,21.0,82132.000000,21210001.0,30.000000,1.000000,1.0,1.000000,70.000000,38.000000,199.000000,4.388845e+06,300.000000,200.000000,200.000000,45.000000,3.000000,1.000000,0.000000,1.000000,1.000000,22000.000000,5000.000000,800.000000,0.000000,0.000000,0.000000,1.000000,2.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,47566.500000,5000.000000,2600.000000,0.000000,0.000000,1.000000
max,21.0,98862.000000,21210001.0,60.000000,10.000000,1.0,2.000000,99.000000,86.000000,500.000000,5.374055e+06,600.000000,500.000000,500.000000,99.000000,20.000000,32.000000,5.000000,7.000000,7.000000,78398.000000,63580.000000,76000.000000,11.000000,2.000000,3.000000,5.000000,32.000000,0.0,10.000000,0.0,0.0,0.0,0.0,0.0,8.000000,1.000000,467048.000000,94670.000000,76541.000000,4.000000,1.000000,1.000000
